In [4]:
import pandas as pd

In [6]:
# required mapping dictionaries

map_payment_method = {
    'UPI' : 'UPI',
    'PhonePe' : 'UPI',
    'GooglePay' : 'UPI',

    'Credit Card' : 'Credit Card',
    'CREDIT_CARD' : 'Credit Card',
    'CC' : 'Credit Card',

    'Cash on Delivery' : 'Cash on Delivery',
    'COD' : 'Cash on Delivery',
    'C.O.D' : 'Cash on Delivery',
    
    'Debit Card' : 'Debit Card',
    'DC' : 'Debit Card',
    'DEBIT_CARD' : 'Debit Card',

    'Internet Banking' : 'Net Banking',
    'INTERNET_BANKING' : 'Net Banking',
    'Net_banking' : 'Net Banking',
    'Net banking' : 'Net Banking',

    'Wallet' : 'Wallet',
    'wallet' : 'Wallet',

    'BNPL' : 'BNPL',
    'bnpl' : 'BNPL'
}

map_category = {
    'Electronics' : 'Electronics',
    'ELECTRONICS' : 'Electronics',
    'Electronics & Accessories' : 'Electronics',
    'Electronic' : 'Electronics',
    'Electronicss' : 'Electronics'
}

map_bool = {
    'No' : False,
    'Yes' : True,

    'False' : False,
    'True' : True,
    'FALSE' : False,
    'TRUE' : True,

    '0' : False,
    '1' : True,

    'No' : False,
    'Yes' : True,
    'N' : False,
    'Y' : True,
    'NO' : False,
    'YES' : True,
    'n' : False,
    'y' : True
}

map_city = {
    'allahabad' : 'Allahabad',
    'prayagraj' : 'Allahabad',

    'kolkata' : 'Kolkata',
    'calcutta' : 'Kolkata',

    'ludhiana' : 'Ludhiana',

    'mumbai' : 'Mumbai',
    'bombay' : 'Mumbai',
    'mumbai/bombay' : 'Mumbai',
    'mumbai /bombay' : 'Mumbai',
    'mumbai/ bombay' : 'Mumbai',
    'mumbai / bombay' : 'Mumbai',
    'mumba' : 'mumbai',

    'delhi ncr' : 'Delhi',
    'new delhi' : 'Delhi',
    'dehli' : 'Delhi',
    'delhi' : 'Delhi',

    'lucknow' : 'Lucknow',
    'lukhnow' : 'Lucknow',

    'bangalore' : 'bengaluru',
    'banglore' : 'bengaluru',
    'bengalore' : 'bengaluru',

    'chenai' : 'chennai',
    'chennai' : 'chennai',
    'madras' : 'chennai',

    'hyderabad' : 'Hyderabad',

    'chandigarh' : 'chandigarh',

    'varanasi' : 'Varanasi',
    'Banaras' : 'Varanasi',
    'Kashi' : 'Varanasi',

    'bareilly' : 'Bareily'
}

In [23]:
# CREATE A FUNCTION TO CLEAN DATA

def data_cleaner(df, map_city, map_category, map_payment_method, map_bool):

    # format and clean order_date column
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='mixed')

    # format and clean original_price_inr column
    df['original_price_inr'] = df['original_price_inr'].astype(str).str.replace(r'[^\d.]', '', regex=True)

    df['original_price_inr'] = pd.to_numeric(df['original_price_inr'], errors='coerce')

    df['original_price_inr'] = df['original_price_inr'].fillna(df['original_price_inr'].median())     # this fills the null values in original price inr column with the median

    # format and clean product ratings column
    df['product_rating'] = df['product_rating'].astype(str)
    df['product_rating'] = df['product_rating'].str.extract(r'(\d+\.?\d*)')
    df['product_rating'] = pd.to_numeric(df['product_rating'])

    # format and customer city ratings column
    df['customer_city'] = df['customer_city'].astype(str)
    df['customer_city'] = df['customer_city'].str.strip().str.lower().replace(map_city).str.title()

    # format and clean boolean columns
    df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
    df['is_prime_member'] = df['is_prime_member'].fillna(False)

    df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
    df['is_prime_eligible'] = df['is_prime_eligible'].fillna(False)

    df['is_festival_sale'] = df['is_festival_sale'].replace(map_bool)
    df['is_festival_sale'] = df['is_festival_sale'].fillna(False)

    # format and clean category column
    df['category'] = df['category'].astype(str).replace(map_category)
    df['category'] = df['category'].fillna(df['category'].mode()[0])

    # format and clean delivery days
    df['delivery_days'] = df['delivery_days'].astype(str).str.strip().str.lower()
    df['delivery_days'] = df['delivery_days'].replace({'express' : '0'})
    df['delivery_days'] = df['delivery_days'].replace({'same day' : '0'})
    df['delivery_days'] = df['delivery_days'].str.extract(r'(\d+)')
    df['delivery_days'] = pd.to_numeric(df['delivery_days'])
    df.loc[df['delivery_days'] > 20, 'delivery_days'] = df['delivery_days'].mode()[0]
    df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].mode()[0])

    # handling duplicate rows

    # strategy- identify duplicate rows, detect genuine repeat orders and data errors, delete any such suspicious row
    # any row with the same transaction id is an absolute error, we remove that
    # any further duplicate row counting more than 2 is sus, we remove that, keeping first
    df = df.drop_duplicates(subset='transaction_id', keep='first')                                                # this drops any row where transaction id is repeated
    dup_cols = ['customer_id','product_name','order_date','original_price_inr']
    counts = df.groupby(dup_cols).transform('size')
    df = df[~((counts > 2) & df.duplicated(subset=dup_cols))]

    # handling price outliers (IQR method)
    # original price is already in numeric
    Q1 = df['original_price_inr'].quantile(0.25)
    Q3 = df['original_price_inr'].quantile(0.75)

    IQR = Q3 - Q1

    upper_limit = Q3 + 1.5 * IQR
    lower_limit = Q1 - 1.5 * IQR

    outliers = df[(df['original_price_inr'] > upper_limit) | 
                    (df['original_price_inr'] < lower_limit)]

    median_price = df['original_price_inr'].median()

    # any values which are more than 10 times the median is pobably an error while entering data
    # so we divide such values by 100
    df.loc[df['original_price_inr'] >= median_price * 10,
                'original_price_inr'] = df['original_price_inr'] / 100

    # format and clean payment method
    df['payment_method'] = df['payment_method'].replace(map_payment_method)

    # drop unnecessary columns
    df = df.drop(columns=['discounted_price_inr', 'subtotal_inr', 'final_amount_inr', 'product_weight_kg'])

    # fill null values in delivery_charges, customer_age_group, festival_name, customer_rating
    df['delivery_charges'] = df['delivery_charges'].fillna(0)                                                  # fill with zero

    df['customer_age_group'] = df['customer_age_group'].fillna(df['customer_age_group'].mode()[0])             # fill with mode

    # format and clean customer ratings column before filling
    df['customer_rating'] = df['customer_rating'].astype(str)
    df['customer_rating'] = df['customer_rating'].str.extract(r'(\d+\.?\d*)')
    df['customer_rating'] = pd.to_numeric(df['customer_rating'])
    df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].mean())                         # fill with mean

    df['festival_name'] = df['festival_name'].fillna('No Festival')

    # re calculate dropped columns that were useful: discounted_price_inr, subtotal_inr, final_amount_inr
    df['discounted_price_inr'] = df['original_price_inr'] * (1 - df['discount_percent']/100)

    df['subtotal_inr'] = df['discounted_price_inr'] * df['quantity']

    df['final_amount_inr'] = df['subtotal_inr'] + df['delivery_charges']

    return df
    

In [26]:
# FOR 2015
# read csv
df_2015 = pd.read_csv("amazon_india_2015.csv")

# clean csv
cleaned_df_2015 = data_cleaner(df_2015, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2015.to_csv("amazon_india_2015_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [27]:
# FOR 2016
# read csv
df_2016 = pd.read_csv("amazon_india_2016.csv")

#clean csv
cleaned_df_2016 = data_cleaner(df_2016, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2016.to_csv("amazon_india_2016_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [28]:
# FOR 2017
# read csv
df_2017 = pd.read_csv("amazon_india_2017.csv")

#clean csv
cleaned_df_2017 = data_cleaner(df_2017, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2017.to_csv("amazon_india_2017_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [29]:
# FOR 2018
# read csv
df_2018 = pd.read_csv("amazon_india_2018.csv")

#clean csv
cleaned_df_2018 = data_cleaner(df_2018, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2018.to_csv("amazon_india_2018_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [30]:
# FOR 2019
# read csv
df_2019 = pd.read_csv("amazon_india_2019.csv")

#clean csv
cleaned_df_2019 = data_cleaner(df_2019, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2019.to_csv("amazon_india_2019_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [31]:
# FOR 2020
# read csv
df_2020 = pd.read_csv("amazon_india_2020.csv")

#clean csv
cleaned_df_2020 = data_cleaner(df_2020, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2020.to_csv("amazon_india_2020_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [32]:
# FOR 2021
# read csv
df_2021 = pd.read_csv("amazon_india_2021.csv")

#clean csv
cleaned_df_2021 = data_cleaner(df_2021, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2021.to_csv("amazon_india_2021_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [33]:
# FOR 2022
# read csv
df_2022 = pd.read_csv("amazon_india_2022.csv")

#clean csv
cleaned_df_2022 = data_cleaner(df_2022, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2022.to_csv("amazon_india_2022_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [34]:
# FOR 2023
# read csv
df_2023 = pd.read_csv("amazon_india_2023.csv")

#clean csv
cleaned_df_2023 = data_cleaner(df_2023, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2023.to_csv("amazon_india_2023_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [35]:
# FOR 2024
# read csv
df_2024 = pd.read_csv("amazon_india_2024.csv")

#clean csv
cleaned_df_2024 = data_cleaner(df_2024, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2024.to_csv("amazon_india_2024_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [36]:
# FOR 2025
# read csv
df_2025 = pd.read_csv("amazon_india_2025.csv")

#clean csv
cleaned_df_2025 = data_cleaner(df_2025, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save csv
cleaned_df_2025.to_csv("amazon_india_2025_cleaned.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_member'] = df['is_prime_member'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_prime_eligible'] = df['is_prime_eligible'].replace(map_bool)
C:\Users\abdul\AppData\Local\Temp\ipykernel_6960\1380007378.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To reta

In [24]:
# FOR COMBINED DATA
# combined csv
df_merged = pd.concat([df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025], ignore_index=True)

# clean combined csv
cleaned_df_merged = data_cleaner(df_merged, map_category=map_category, map_bool=map_bool, map_city=map_city, map_payment_method=map_payment_method)

# save combined csv
cleaned_df_merged.to_csv("amazon_india_2015_2025_merged_cleaned.csv", index=False)